# Thảo luận, phân tích lỗi và kết luận

## 1. Tổng hợp kết quả thực nghiệm trên 25.000 bài báo

1. **Thu thập dữ liệu và kiểm toán nguồn (Milestone 1)**: Sau khi hoàn tất phiên chạy bù `--resume` trên Google Colab CPU-only cho phân vùng 24120470, kho dữ liệu hợp nhất đạt đủ 25.000/25.000 bài báo duy nhất (không trùng chéo giữa 5 thành viên), gồm 41.498 phiên bản (xác minh hợp lệ 41.487 phiên bản; lọc bỏ tệp hình ảnh trên luồng giải nén giúp giảm 89,30% dung lượng, còn 22,72 GiB), 976.351 dòng trích dẫn ở tầng vận hành (0 trích dẫn mồ côi) và 456.149 cạnh có đích arXiv trong bảng `references`.
2. **Phân tích cấu trúc LaTeX và khử trùng xuyên phiên bản (Milestone 2)**: Trên 38.554 phiên bản có mã nguồn TeX xác minh, trình phân tích đạt 33.025 phiên bản thành công hoàn toàn (85,66%) và 4.503 phiên bản bán phần (tổng cộng 37.528 phiên bản dựng được cây cấu trúc, đạt 97,34%), chỉ còn 1.026 phiên bản (2,66%) gặp lỗi cú pháp nặng; 2.764 phiên bản `empty_no_tex` khớp chính xác với kết quả kiểm toán M1. Khử trùng hai tầng giúp giảm 35,04% số dòng bảng `elements` (từ 23.099.589 xuống 15.006.000 dòng; giảm 46,95% trên nhóm bài đa phiên bản) và giảm 35,83% số dòng bảng `bibtex` (từ 1.834.650 xuống 1.177.293 dòng; giảm 50,41% trên nhóm bài đa phiên bản).
3. **Kiểm chuẩn toàn cục và tài nguyên tính toán**: Công cụ `validate_project` xác nhận `structural_valid = True` (0 lỗi khóa chính, khóa ngoại hoặc chu trình trên 22.532.299 cạnh `hierarchy`) với mức bộ nhớ RAM trung bình 240,00 MiB (đỉnh 845,34 MiB) và tổng thời gian chạy 5 phiên trên Google Colab CPU-only là 18.692,09 giây (trung bình 0,7477 giây/bài).

## 2. Phân tích 15 mẫu thông điệp chẩn đoán phổ biến nhất

Bảng và biểu đồ dưới đây tách rõ **số phiên bản phân biệt bị ảnh hưởng** (`distinct_versions`) và **tổng số lượt cảnh báo phát sinh** (`occurrences`) trên 41.498 phiên bản, đồng thời thay thế tên tệp cụ thể bằng nhãn khái quát:

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
top_d = pd.DataFrame(m2["top_diagnostics"])
top_d["Tỷ lệ / 41.498 bản (%)"] = 100.0 * top_d["distinct_versions"] / scope["versions"]
top_view = top_d.rename(columns={
    "message": "Mẫu thông điệp chẩn đoán",
    "distinct_versions": "Số phiên bản phân biệt",
    "occurrences": "Tổng số lượt cảnh báo",
    "primary_status": "Trạng thái chính",
})[["Mẫu thông điệp chẩn đoán", "Số phiên bản phân biệt", "Tỷ lệ / 41.498 bản (%)", "Tổng số lượt cảnh báo", "Trạng thái chính"]]
display(style_table(
    top_view,
    caption="Bảng 12.1. Mười lăm mẫu thông điệp chẩn đoán xuất hiện nhiều nhất trên 41.498 phiên bản.",
    int_cols=["Số phiên bản phân biệt", "Tổng số lượt cảnh báo"],
    pct_cols=["Tỷ lệ / 41.498 bản (%)"],
))

top10 = top_d.head(10).iloc[::-1]
short_lbl = [str(m)[:44] + ("..." if len(str(m)) > 44 else "") for m in top10["message"]]
yt = np.arange(len(top10))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.0), sharey=True)
ax1.barh(yt, top10["distinct_versions"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], height=0.58)
ax1.set_yticks(yt)
ax1.set_yticklabels(short_lbl, fontsize=8.2)
ax1.set_xlim(0, float(top10["distinct_versions"].max()) * 1.25)
ax1.grid(axis="x")
ax1.set_xlabel("Số phiên bản phân biệt bị ảnh hưởng")
ax1.set_title("Hình 12.1a. Theo số phiên bản phân biệt")
for idx_v, v_v in enumerate(top10["distinct_versions"]):
    ax1.text(float(v_v) + 18, idx_v, fmt_int(v_v), va="center", fontsize=7.9)

ax2.barh(yt, top10["occurrences"].to_numpy(dtype=float), color=BOOK_PALETTE["steel"], height=0.58)
ax2.set_xscale("log")
ax2.set_xlim(10, 25000)
ax2.grid(axis="x")
ax2.set_xlabel("Tổng số lượt cảnh báo phát sinh (thang log10)")
ax2.set_title("Hình 12.1b. Theo tổng số lượt xuất hiện")
for idx_o, v_o in enumerate(top10["occurrences"]):
    ax2.text(float(v_o) * 1.15, idx_o, fmt_int(v_o), va="center", fontsize=7.9)
plt.tight_layout()
plt.show()

- **`item outside supported list` (9.087 lượt cảnh báo trên 791 phiên bản phân biệt, trạng thái `partial`)**: Xuất hiện khi tác giả đặt nhiều lệnh `\item` bên trong các môi trường danh sách hoặc khung thuật toán tự định nghĩa bằng macro riêng ngoài các môi trường chuẩn (`itemize`, `enumerate`, `description`, `compactitem`, `inparaenum`, `list`, `trivlist`). Trình phân tích chuyển 791 phiên bản này (1,91% tổng số phiên bản) sang trạng thái `partial` để giữ lại toàn bộ phần cây cấu trúc hợp lệ còn lại.
- **`Unclosed TeX group` (285 phiên bản, trạng thái `error`) và `unmatched end environment: (môi trường)` (331 lượt trên 90 phiên bản phân biệt)**: Phát sinh do dấu ngoặc nhọn không cân bằng hoặc đóng/mở môi trường tùy biến không khớp trong mã nguồn gốc của tác giả.
- **Các cảnh báo về tệp tham khảo và tệp con**: Gồm `missing include: (đường dẫn tương đối)` (3.477 lượt trên 805 phiên bản), `missing/unsafe bibliography: (tệp .bib)` (765 lượt trên 637 phiên bản khai báo tệp `.bib` không đính kèm trong gói nộp), `unsupported/empty compiled bibliography: (tệp .bbl)` (621 phiên bản), `Expected one compilation root` (614 phiên bản) và `dynamic/unsupported include in (tệp .tex)` (574 lượt trên 271 phiên bản dùng biến macro trong đường dẫn `\input`).

## 3. Liên kết với giai đoạn khớp trích dẫn (Milestone 3) và bộ dữ liệu công khai

```{note}
**Ứng dụng trực tiếp đầu ra M1 - M2 vào Milestone 3 và kho dữ liệu 6 bảng trên Hugging Face:**
- **Khớp trích dẫn sử dụng ngữ cảnh cây cấu trúc M2**: Nhờ đồng bộ khóa `\cite{}` xuyên phiên bản ở M2, 98,05% mục BibTeX trong thực nghiệm Milestone 3 truy vết được câu văn trích dẫn và tiêu đề mục cha từ hai bảng `elements` và `hierarchy`. Bộ 25 đặc trưng học máy truyền thống (không dùng LLM hoặc mô hình ngôn ngữ tiền huấn luyện) huấn luyện trên 3.022 bài báo gán nhãn (25 bài thủ công gồm 1.150 mục và 2.997 bài tự động gồm 161.319 mục, tổng cộng 162.469 dòng trong bảng `predictions`) giúp mô hình LightGBM Ranker đạt trần lý thuyết MRR@5 = 0,6907 trên tập Test cố định (373/540 mục; tách riêng Test-Manual đạt 0,8885 và Test-Auto đạt 0,5071). Trong kịch bản ẩn tiêu đề `.bbl`, nhóm đặc trưng ngữ cảnh cây M2 nâng kết quả lên 0,6802 trên tập Test và 0,8530 ± 0,0163 trên kiểm định chéo 5-fold tập Manual.
- **Bộ dữ liệu 6 bảng Parquet trên Hugging Face (38 phân mảnh, 39.359.210 dòng, 3,92 GiB)**: Xuất bản tại [nquocdat06/arxiv-harvester-data](https://huggingface.co/datasets/nquocdat06/arxiv-harvester-data). Như đã đối chiếu tại Bảng 8.4, việc gộp 75.000 tệp Parquet nhỏ của M2 (3,89 GiB thuần; 3,93 GiB nếu tính cả 3 tệp JSON quản trị trong thư mục `parsed/`) thành 35 phân mảnh lớn nén Snappy giúp giảm 55,33 MiB chi phí lưu trữ, nên trọn bộ 6 bảng (`papers`, `references`, `elements`, `hierarchy`, `bibtex`, `predictions`) có tổng dung lượng 3,92 GiB (4.207.534.396 byte).
- **Mã nguồn và notebook phân tích M3**: Xem tại [NguyenQuocDat06-AI/arxiv-harvester](https://github.com/NguyenQuocDat06-AI/arxiv-harvester) và [`docs/09_m3_feature_eda_and_matching.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/docs/09_m3_feature_eda_and_matching.ipynb).
```

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/validation.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).
- Mã nguồn triển khai: [`tools/package_hf_dataset.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/package_hf_dataset.py).
- Mã nguồn triển khai: [`docs/09_m3_feature_eda_and_matching.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/docs/09_m3_feature_eda_and_matching.ipynb).
- Mã nguồn triển khai: [`tools/build_team_25000_book.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/build_team_25000_book.py).